# 1) The AC power flow on a real network — Exercises

**Goals:** count the equations, test the Jacobian against finite differences, see what a simplified Jacobian does to convergence, add generator reactive-power limits, and find how much those limits lower the loadability of the network.

The setup cell holds `case14_data(load_scale, enforce_q_lims)` (the IEEE 14-bus system solved by pandapower, with its internal Y-bus, injections `S`, start voltages `V0`, bus lists and ordering), the lecture's `analytic_jacobian` and `newton_raphson`. Everything is in per unit on 100 MVA unless stated.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import logging
import warnings

import numpy as np
import pandas as pd
import pandapower as pp
import pandapower.networks as pn

warnings.filterwarnings("ignore")
logging.getLogger("pandapower").setLevel(logging.ERROR)


def case14_data(load_scale: float = 1.0, enforce_q_lims: bool = False):
    """IEEE 14-bus system with all loads scaled, solved by pandapower. Returns the net and its internal data."""
    net = pn.case14()
    net.load[["p_mw", "q_mvar"]] *= load_scale
    pp.runpp(net, init="flat", numba=False, tolerance_mva=1e-10, enforce_q_lims=enforce_q_lims)
    internal = net._ppc["internal"]
    order = net._pd2ppc_lookups["bus"]
    V0 = np.ones(len(internal["Sbus"]), dtype=complex)
    for table in (net.gen, net.ext_grid):
        V0[order[table.bus.to_numpy()]] = table.vm_pu.to_numpy()
    return net, {"Y": internal["Ybus"].toarray(), "S": internal["Sbus"].copy(), "V0": V0, "ref": internal["ref"],
                 "pv": internal["pv"], "pq": internal["pq"], "order": order, "base": net._ppc["baseMVA"]}


def analytic_jacobian(Y, V, pv, pq):
    """The lecture's polar Jacobian: rows P(pv+pq), Q(pq); columns angle(pv+pq), magnitude(pq)."""
    pvpq = np.r_[pv, pq]
    I = Y @ V
    Vn = V / np.abs(V)
    dS_dVa = 1j * np.diag(V) @ np.conj(np.diag(I) - Y @ np.diag(V))
    dS_dVm = np.diag(V) @ np.conj(Y @ np.diag(Vn)) + np.conj(np.diag(I)) @ np.diag(Vn)
    return np.block([[dS_dVa[np.ix_(pvpq, pvpq)].real, dS_dVm[np.ix_(pvpq, pq)].real],
                     [dS_dVa[np.ix_(pq, pvpq)].imag, dS_dVm[np.ix_(pq, pq)].imag]])


def newton_raphson(Y, S, V0, ref, pv, pq, tol=1e-10, max_iter=30):
    """The lecture's solver: complex voltages and the largest mismatch per iteration."""
    V = V0.astype(complex).copy()
    pvpq = np.r_[pv, pq]
    history = []
    for _ in range(max_iter + 1):
        mismatch = V * np.conj(Y @ V) - S
        F = np.r_[mismatch[pvpq].real, mismatch[pq].imag]
        history.append(np.abs(F).max())
        if history[-1] < tol:
            return V, history
        dx = np.linalg.solve(analytic_jacobian(Y, V, pv, pq), -F)
        Va, Vm = np.angle(V), np.abs(V)
        Va[pvpq] += dx[:len(pvpq)]
        Vm[pq] += dx[len(pvpq):]
        V = Vm * np.exp(1j * Va)
    raise RuntimeError(f"no convergence in {max_iter} iterations")

### Warm-ups

**1. Equations and injections**

(a) Write `count_equations(n_pv, n_pq)` returning (number of equations, unknown angles, unknown voltage magnitudes).
(b) Write `injections(V, Y)` returning the complex power injected at every bus. Check: at pandapower's solution, the injections equal the specified ones in P at PV and PQ buses and in Q at PQ buses.

In [ ]:
def count_equations(n_pv: int, n_pq: int) -> tuple[int, int, int]:
    """(equations, unknown angles, unknown magnitudes) of the AC power flow."""
    ...  # YOUR CODE HERE


def injections(V: np.ndarray, Y: np.ndarray) -> np.ndarray:
    """Complex power injected at each bus (p.u.)."""
    ...  # YOUR CODE HERE

In [ ]:
net, d = case14_data()
assert count_equations(len(d["pv"]), len(d["pq"])) == (22, 13, 9)
V_pp = net._ppc["internal"]["V"]                      # pandapower's solution, internal order
S_pp = injections(V_pp, d["Y"])
pvpq = np.r_[d["pv"], d["pq"]]
assert np.abs(S_pp[pvpq].real - d["S"][pvpq].real).max() < 1e-9 and np.abs(S_pp[d["pq"]].imag - d["S"][d["pq"]].imag).max() < 1e-9
print("slack injection (MW, Mvar):", np.round(S_pp[d["ref"][0]] * d["base"], 2))

**2. Is the Jacobian right?**

Write `jacobian_fd(Y, S, V, pv, pq, eps=1e-7)`: the Jacobian by **finite differences** — perturb each angle (PV and PQ buses) and each magnitude (PQ buses) by `eps` and record how the mismatch vector changes. Compare it with `analytic_jacobian` at the flat start and at the solution.

In [ ]:
def jacobian_fd(Y, S, V, pv, pq, eps: float = 1e-7) -> np.ndarray:
    """Jacobian of the mismatch [P(pv+pq), Q(pq)] w.r.t. [angle(pv+pq), magnitude(pq)] by forward differences."""
    ...  # YOUR CODE HERE

In [ ]:
for label, V in {"flat start": d["V0"], "solution": V_pp}.items():
    gap = np.abs(jacobian_fd(d["Y"], d["S"], V, d["pv"], d["pq"]) - analytic_jacobian(d["Y"], V, d["pv"], d["pq"])).max()
    assert gap < 1e-4, label
    print(f"{label}: largest difference {gap:.1e}")

### Core

**3. A simplified Jacobian**

Fast power-flow methods drop the weak couplings between angles and reactive power, and between voltage magnitudes and active power. Write `decoupled_newton(Y, S, V0, ref, pv, pq)`: each iteration solves only the ∂P/∂θ block for the angle update and the ∂Q/∂|V| block for the magnitude update (both taken from `analytic_jacobian`). Compare its iterations and its mismatch history with full Newton–Raphson, and check that it reaches the same solution.

In [ ]:
def decoupled_newton(Y, S, V0, ref, pv, pq, tol=1e-10, max_iter=100):
    """Newton-Raphson with only the dP/dangle and dQ/d|V| blocks. Returns V and the mismatch history."""
    ...  # YOUR CODE HERE

In [ ]:
V_full, h_full = newton_raphson(d["Y"], d["S"], d["V0"], d["ref"], d["pv"], d["pq"])
V_dec, h_dec = decoupled_newton(d["Y"], d["S"], d["V0"], d["ref"], d["pv"], d["pq"])
assert np.abs(V_dec - V_full).max() < 1e-8, "same solution"
assert len(h_dec) > len(h_full), "the simplified Jacobian needs more iterations"
ratios = np.array(h_dec[2:]) / np.array(h_dec[1:-1])
print(f"full Newton: {len(h_full) - 1} iterations; decoupled: {len(h_dec) - 1} iterations; "
      f"decoupled mismatch shrinks by a factor of about {np.median(ratios):.2f} per step")
print("full:      ", "  ".join(f"{h:.0e}" for h in h_full))
print("decoupled: ", "  ".join(f"{h:.0e}" for h in h_dec[:8]), " ...", f"{h_dec[-1]:.0e} after {len(h_dec) - 1} iterations")

**4. Generator reactive-power limits**

At 1.3 times the base load some generators would have to produce more reactive power than they can. Write `solve_with_q_limits(load_scale)`: solve; compute each generator's reactive output (MVAr) — the bus's reactive injection plus its local load; if a generator is outside its limits, fix its output at the limit and turn its bus from PV into PQ; re-solve until no limit is violated. Return the voltages (in pandapower bus order), the generators that hit a limit and the number of rounds. Check against pandapower with `enforce_q_lims=True`.

In [ ]:
def solve_with_q_limits(load_scale: float) -> dict:
    """Own power flow with PV-to-PQ switching at generator reactive limits."""
    ...  # YOUR CODE HERE

In [ ]:
own = solve_with_q_limits(1.3)
reference, _ = case14_data(1.3, enforce_q_lims=True)
assert np.abs(own["vm_pu"].to_numpy() - reference.res_bus.vm_pu.to_numpy()).max() < 1e-8
no_limits, _ = case14_data(1.3)
print("generators at their limit (bus: Mvar):", own["at_limit_mvar"], "after", own["rounds"], "rounds")
pd.DataFrame({"without limits": no_limits.res_bus.vm_pu.to_numpy(), "with limits": own["vm_pu"].to_numpy()},
             index=own["vm_pu"].index).T.round(3)

**5. Interpretation**

What does the decoupled method's mismatch history tell you about its convergence rate, and why does it still reach the exact solution? What happened to the voltages when the generators reached their reactive limits, and why?

*Your answer here.*

### Challenge

**6. How much do reactive limits cost in loadability?**

Find the largest load scale (to 0.01) at which the power flow still converges — by bisection — once **without** reactive limits (scaling only the loads, generators' active power unchanged) and once **with** them (`solve_with_q_limits`). Treat a failed solve as "beyond the limit". Write `loadability(with_limits)`.

In [ ]:
def loadability(with_limits: bool, low: float = 1.0, high: float = 6.0, tol: float = 0.01) -> float:
    """Largest load scale that still solves, by bisection."""
    ...  # YOUR CODE HERE

In [ ]:
without = loadability(False)
with_q = loadability(True)
assert with_q < without, "reactive limits lower the loadability"
print(f"loadability without reactive limits: {without:.2f} x base load; with limits: {with_q:.2f} x")

**7. Interpretation**

How large is the difference, and what does it say about studies that ignore generator reactive limits? Which kind of real-world equipment would raise the limit again?

*Your answer here.*